In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 


In [ ]:
import os

# Caminho para salvar o gráfico e o arquivo CSV
save_path = '../../results/regression/graphics'
os.makedirs(save_path, exist_ok=True)  # Cria o diretório, caso não exista

# Filtragem e visualização
bbr = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
columns_to_drop = ['PolynomialRegression', 'AdaBoostRegressor', 'ElasticNet', 
                   'LinearRegression', 'MLPRegressor', 'SVR', 'KNeighborsRegressor']

sources_to_drop = ['df', 'rj', 'sp', 'pa', 'sc', 'pr', 'mg'] #lembrar de semre excluir o df, pois o dataset ficou pequeno 

bbr = bbr.drop(columns=columns_to_drop, errors='ignore')
#bbr = bbr[~bbr['source'].isin(sources_to_drop)]

models = bbr.columns[1:]
y = np.arange(len(bbr['source']))  # Agora as categorias estão no eixo y
height = 0.7 / len(models)         # Ajustando altura das barras para maior largura - 0.7

custom_colors = [
    'blue', 'green', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'maroon', 'brown'
    #'olive', 'peru', 'gold', 'deeppink', 'lime', 'royalblue', 'darkviolet'
]

# Ajustando o tamanho da figura para ser menor em largura e maior em comprimento
fig, ax = plt.subplots(figsize=(10, 12))  

for i, model in enumerate(models):
    # Usando as cores definidas manualmente
    ax.barh(y + i * height, bbr[model], height, label=model, color=custom_colors[i], edgecolor='black')  # Adicionando borda preta nas barras

ax.set_title("Comparação de RMSE por modelo de regressão", fontsize=14)
ax.set_xlabel("NRMSE", fontsize=12)  # Eixo x agora representa os valores
ax.set_ylabel("Source", fontsize=12)  # Eixo y representa as categorias
ax.set_yticks(y + height * (len(models) / 2 - 0.5))
ax.set_yticklabels(bbr['source'].str.upper())

# Ajustando o espaço abaixo do gráfico para a legenda
plt.subplots_adjust(bottom=0.15)

# Legenda ajustada para ficar mais próxima do gráfico
ax.legend(
    title="Modelos de Regressao",
    bbox_to_anchor=(0.5, -0.05),  # Ajusta a posição para ficar mais próxima
    loc='upper center',
    ncol=3
)

ax.grid(axis='x', linestyle='--', alpha=0.7)  # Grid no eixo x
plt.tight_layout()

# Salvar o gráfico
#graph_path = os.path.join(save_path, '5-modelos-todos-links-ruins.png')
#plt.savefig(graph_path, dpi=300)
#print(f"Gráfico salvo em: {graph_path}")

plt.show()



# Métricas estatísticas:
#                             nrmse                                
#                              mean     std  median     min     max
# model                                                            
# AdaBoostRegressor          0.2676  0.3231  0.0591  0.0359  0.9362
# CatBoostRegressor          0.2224  0.2615  0.0511  0.0314  0.7054
# ElasticNet                 0.2934  0.3607  0.0609  0.0379  1.0186
# GradientBoostingRegressor  0.2310  0.2735  0.0512  0.0315  0.7379
# KNeighborsRegressor        0.2470  0.2870  0.0570  0.0358  0.7750
# LGBMRegressor              0.2241  0.2627  0.0513  0.0318  0.7195
# LinearRegression           0.2934  0.3607  0.0609  0.0379  1.0186
# MLPRegressor               0.3439  0.3611  0.0924  0.0379  1.0381
# PolynomialRegression       0.2711  0.3258  0.0560  0.0360  0.9310
# RandomForestRegressor      0.2275  0.2671  0.0518  0.0317  0.7224
# SVR                        0.3548  0.4437  0.0657  0.0399  1.2578
# XGBRegressor               0.2285  0.2701  0.0520  0.0316  0.7304

In [ ]:
df1 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_bbr.csv')
df2 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_cubic.csv')

In [ ]:
# quero percorrer uma pasta que contem varios csv (rferentes a minha source), 
# cada arquivo tem uma coluna de y_test e as outras colunas sao os valores de y_pred para cada modelo
# quero categorizar cada y_test e y_pred (modelo por modelo) e verificar se eles estao na mesma categoria e salva isso em porcentagem 
# de acertos. quetro que faca isso recursivamente para todos os meus arquivose links. 
#apos isso quero plotar um grafico de barras em que as barras estarao juntas por source (cada source vai ter barras comparando o modelo)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# def categorize_nrmse(value):
#     if value < 0.1:
#         return 'Excelente'
#     elif value < 0.2:
#         return 'Bom'
#     elif value < 0.3:
#         return 'Razoável'
#     elif value < 0.5:
#         return 'Ruim'
#     else:
#         return 'Muito Ruim'
def categorize_nrmse(value):
    if value < 0:
        return 'Valor Inválido'
    elif value < 0.1:
        return 'Excelente'
    elif value < 0.2:
        return 'Bom'
    elif value < 0.3:
        return 'Razoável'
    elif value < 0.5:
        return 'Ruim'
    else:
        return 'Muito Ruim'


def analyze_nrmse(df):
    df_melted = df.melt(id_vars=['source'], var_name='model', value_name='nrmse')
    
    # Adicionar categorias
    df_melted['categoria'] = df_melted['nrmse'].apply(categorize_nrmse)
    
    # 1. Análise geral por modelo
    model_analysis = df_melted.groupby('model').agg({
        'nrmse': ['mean', 'std', 'median', 'min', 'max']
    }).round(4)
    
    # 2. Distribuição das categorias por modelo
    category_dist = pd.crosstab(df_melted['model'], df_melted['categoria'], normalize='index') * 100
    
    # 3. Identificar melhores estados por modelo
    best_states = df_melted.loc[df_melted.groupby('model')['nrmse'].idxmin()]
    
    # 4. Identificar piores estados por modelo
    worst_states = df_melted.loc[df_melted.groupby('model')['nrmse'].idxmax()]
    
    return df_melted, model_analysis, category_dist, best_states, worst_states

def plot_analysis(df_melted, model_analysis, category_dist):
    """
    Cria visualizações para a análise do NRMSE.
    """
    plt.style.use('default')
    
    fig = plt.figure(figsize=(20, 15))
    
    # # 1. Boxplot dos modelos
    # plt.subplot(2, 2, 1)
    # sns.boxplot(data=df_melted, x='model', y='nrmse', width=0.7)
    # plt.xticks(rotation=45, ha='right')
    # plt.title('Distribuição do NRMSE por Modelo')
    # plt.xlabel('Modelo')
    # plt.ylabel('NRMSE')
    
    # 2. Heatmap da distribuição de categorias
    plt.subplot(2, 2, 2)
    sns.heatmap(category_dist, annot=True, fmt='.1f', cmap='YlOrRd')
    plt.title('Distribuição das Categorias por Modelo (%)')
    plt.xlabel('Categoria')
    plt.ylabel('Modelo')
    
    # # 3. Gráfico de barras do NRMSE médio
    # plt.subplot(2, 2, 3)
    # model_means = model_analysis['nrmse']['mean'].sort_values()
    # plt.bar(range(len(model_means)), model_means)
    # plt.xticks(range(len(model_means)), model_means.index, rotation=45, ha='right')
    # plt.title('NRMSE Médio por Modelo')
    # plt.xlabel('Modelo')
    # plt.ylabel('NRMSE Médio')
    
    # 4. Gráfico de dispersão do NRMSE por estado
    plt.subplot(2, 2, 4)
    markers = ['o', 's', '^', 'v', 'D', 'p', 'h', '8', '*', '+', 'x', 'd']
    for i, model in enumerate(df_melted['model'].unique()):
        model_data = df_melted[df_melted['model'] == model]
        plt.scatter(model_data['source'], model_data['nrmse'], 
                   label=model, alpha=0.6, marker=markers[i % len(markers)])
    plt.xticks(rotation=45, ha='right')
    plt.title('NRMSE por Estado e Modelo')
    plt.xlabel('Estado')
    plt.ylabel('NRMSE')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    return fig


df = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')


df_melted, model_analysis, category_dist, best_states, worst_states = analyze_nrmse(df)

fig = plot_analysis(df_melted, model_analysis, category_dist)

# Imprimir resultados detalhados
print("\n=== Análise por Modelo ===")
print("\nMétricas estatísticas:")
print(model_analysis)

print("\n=== Melhores Estados por Modelo ===")
best_states_formatted = best_states[['model', 'source', 'nrmse']].sort_values('nrmse')
print(best_states_formatted.to_string())

print("\n=== Piores Estados por Modelo ===")
worst_states_formatted = worst_states[['model', 'source', 'nrmse']].sort_values('nrmse', ascending=False)
print(worst_states_formatted.to_string())

# Calcular e mostrar distribuição geral das categorias
total_dist = df_melted['categoria'].value_counts(normalize=True) * 100
print("\n=== Distribuição Geral das Categorias ===")
print(total_dist.round(2).sort_index())

# Salvar resultados em arquivos
output_dir = '../../results/regression/analysis'
os.makedirs(output_dir, exist_ok=True)

model_analysis.to_csv(f'{output_dir}/model_analysis_bbr.csv')
category_dist.to_csv(f'{output_dir}/category_distribution_bbr.csv')
plt.savefig(f'{output_dir}/nrmse_analysis_plots_bbr.png', bbox_inches='tight', dpi=300)

# Salvar também um resumo em formato mais legível
with open(f'{output_dir}/analysis_summary_bbr.txt', 'w') as f:
    f.write("=== Análise de NRMSE ===\n\n")
    f.write("Distribuição das Categorias:\n")
    f.write(total_dist.round(2).sort_index().to_string())
    f.write("\n\nMelhores Estados:\n")
    f.write(best_states_formatted.to_string())
    f.write("\n\nPiores Estados:\n")
    f.write(worst_states_formatted.to_string())

In [ ]:
# Coeficiente de variância dos valores de vazao 
# Here I combine datasets with the same source (e.g., ce-*)
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            #print(f"Processing files with source: {source}")
            df = combine_datasets(path, source)
            
            if df is not None:
                key_name = f"{source.strip('-')}" 
                dataframes_by_source[key_name] = df
dataframes_by_source.pop('df', None) #df tem poucas linhas, entao a gente resolveu descatra isso 

protocol = 'Vazao_bbr'
rmse = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
rmse = rmse[~(rmse['source'] == 'df')]

correlations_by_model = {}
variances = {}

for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
    variance_coef = (dataset[protocol].std() / dataset[protocol].mean()) * 100
    variances[key] = variance_coef

models = rmse.columns[1:]

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])  
    })
    correlation = df_model[['RMSE', 'Variance_coef']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

# Exibindo as correlações
print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model}: {correlation:.4f}")




# Convertendo o dicionário em DataFrame
correlations_df = pd.DataFrame(list(correlations_by_model.items()), 
                             columns=['Model', 'Correlation'])
correlations_df = correlations_df.sort_values('Correlation', ascending=True)

# Configurando o tema do seaborn
sns.set_theme(style="whitegrid")

# Criando os gráficos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. Gráfico de barras horizontal usando barplot do seaborn
sns.barplot(y='Model', x='Correlation', data=correlations_df, ax=ax1,
           palette='viridis', orient='h')
ax1.set_title('Correlation between RMSE and Dataset Variance by Model')
ax1.set_xlabel('Correlation Coefficient')

# 2. Heatmap das correlações
heatmap_data = correlations_df.set_index('Model')
sns.heatmap(heatmap_data.T, annot=True, cmap='RdYlBu', center=0, 
            fmt='.3f', ax=ax2)
ax2.set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

# Scatter plots para cada modelo
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for idx, model in enumerate(correlations_by_model.keys()):
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])
    })
    
    sns.scatterplot(data=df_model, x='Variance_coef', y='RMSE', ax=axes[idx])
    axes[idx].set_title(f'{model}\nCorr: {correlations_by_model[model]:.3f}')
    axes[idx].set_xlabel('Variance Coefficient (%)')
    axes[idx].set_ylabel('RMSE')

plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(12, 6))
rmse_melted = rmse.melt(id_vars=['source'], 
                        var_name='Model', 
                        value_name='RMSE')
sns.boxplot(x='Model', y='RMSE', data=rmse_melted, palette='viridis')
plt.xticks(rotation=45)
plt.title('Distribution of RMSE by Model')
plt.tight_layout()
plt.show()